# Objective

The main objective of this HR Analytics project is to explore and analyze the dataset for "Atlas Labs". The project aims to preprocess, and visualize the data using Tableau to create a clean, self-explanatory, and interactive dashboard for the HR team of Atlas Labs. The dashboard will provide valuable insights into employee attrition and the factors that impact attrition.

Data preprocessing has been performed in this Google Colab, including data cleaning, data wrangling, and anomaly detection. Ensuring data quality is crucial to obtaining reliable and accurate results from the analysis. As the saying goes, "*garbage in, garbage out*", meaning that the conclusions drawn from data analysis can only be as reliable as the data used. Hence, the project emphasizes the importance of using clean, accurate, and reliable data to derive meaningful and trustworthy insights.

By combining the power of data analysis and visualization, this HR Analytics project seeks to provide valuable and actionable insights that will contribute to fostering a positive work environment, optimizing talent management strategies, and ultimately improving employee satisfaction and retention at Atlas Labs.

# Data Preparation

In [92]:
# import pandas
import pandas as pd

In [93]:
# import datasets
employee = pd.read_csv('Datasets/employee.csv', parse_dates=['HireDate'])
performance_rating = pd.read_csv('Datasets/performance_rating.csv', delimiter=';', parse_dates=['ReviewDate'])
education_level = pd.read_csv('Datasets/education_level.csv')

# Create a true employee-grain dataset by merging Employee and Education Level
# Add a deterministic EmployeeRecordID based on the stable source-row order.
employee['EmployeeRecordID'] = range(1, len(employee) + 1)

employee_base = employee.merge(
    education_level,
    left_on='Education',
    right_on='EducationLevelID',
    how='left',
    validate='many_to_one'
)

In [94]:
performance_rating.head()

,PerformanceID,EmployeeID,ReviewDate,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating
0,PR01,79F7-78EC,2013-01-02,Very Satisfied,Satisfied,Very Satisfied,1,0,Satisfied,Exceeds Expectation,Exceeds Expectation
1,PR02,B61E-0F26,2013-01-03,Very Satisfied,Satisfied,Satisfied,1,3,Satisfied,Exceeds Expectation,Meets Expectation
2,PR03,F5E3-48BB,2013-01-03,Neutral,Satisfied,Very Satisfied,3,2,Neutral,Above and Beyond,Exceeds Expectation
3,PR04,0678-748A,2013-01-04,Very Satisfied,Neutral,Dissatisfied,2,0,Dissatisfied,Meets Expectation,Needs Improvement
4,PR05,541F-3E19,2013-01-04,Very Satisfied,Dissatisfied,Neutral,1,0,Satisfied,Exceeds Expectation,Meets Expectation


In [95]:
employee.head()

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,...,MaritalStatus,Salary,StockOptionLevel,OverTime,HireDate,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,Divorced,102059,1,No,2012-01-03,No,10,4,9,7
1,CBCB-9C9D,Leonerd,Aland,Male,38,Some Travel,Sales,23,CA,White,...,Single,157718,0,Yes,2012-01-04,No,10,6,10,0
2,95D7-1CE9,Ahmed,Sykes,Male,43,Some Travel,Human Resources,29,CA,Asian or Asian American,...,Married,309964,1,No,2012-01-04,No,10,6,10,8
3,47A0-559B,Ermentrude,Berrie,Non-Binary,39,Some Travel,Technology,12,IL,White,...,Married,293132,0,No,2012-01-05,No,10,10,10,0
4,42CC-040A,Stace,Savege,Female,29,Some Travel,Human Resources,29,CA,White,...,Single,49606,0,No,2012-01-05,Yes,6,1,1,6


In [96]:
education_level

,EducationLevelID,EducationLevel
0,1,No Formal Qualifications
1,2,High School
2,3,Bachelors
3,4,Masters
4,5,Doctorate


# Data Cleaning

## Checking Missing Value

In [97]:
performance_rating.isna().sum()

PerformanceID                      0
EmployeeID                         0
ReviewDate                         0
EnvironmentSatisfaction            0
JobSatisfaction                    0
RelationshipSatisfaction           0
TrainingOpportunitiesWithinYear    0
TrainingOpportunitiesTaken         0
WorkLifeBalance                    0
SelfRating                         0
ManagerRating                      0
dtype: int64

In [98]:
employee.isna().sum()

EmployeeID                 0
FirstName                  0
LastName                   0
Gender                     0
Age                        0
BusinessTravel             0
Department                 0
DistanceFromHome (KM)      0
State                      0
Ethnicity                  0
Education                  0
EducationField             0
JobRole                    0
MaritalStatus              0
Salary                     0
StockOptionLevel           0
OverTime                   0
HireDate                   0
Attrition                  0
YearsAtCompany             0
YearsInMostRecentRole      0
YearsSinceLastPromotion    0
YearsWithCurrManager       0
dtype: int64

However, upon inspection, it was found that all three datasets do not contain any missing values. This is a good sign, as it ensures the data's completeness and saves us from the need to handle missing data imputation.

## Duplicated Data

In [99]:
employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   EmployeeID               1470 non-null   object        
 1   FirstName                1470 non-null   object        
 2   LastName                 1470 non-null   object        
 3   Gender                   1470 non-null   object        
 4   Age                      1470 non-null   int64         
 5   BusinessTravel           1470 non-null   object        
 6   Department               1470 non-null   object        
 7   DistanceFromHome (KM)    1470 non-null   int64         
 8   State                    1470 non-null   object        
 9   Ethnicity                1470 non-null   object        
 10  Education                1470 non-null   int64         
 11  EducationField           1470 non-null   object        
 12  JobRole                  1470 non-

In [100]:
employee.nunique()

EmployeeID                 1468
FirstName                  1334
LastName                   1441
Gender                        4
Age                          34
BusinessTravel                3
Department                    3
DistanceFromHome (KM)        45
State                         3
Ethnicity                     7
Education                     5
EducationField                8
JobRole                      13
MaritalStatus                 3
Salary                     1455
StockOptionLevel              4
OverTime                      2
HireDate                   1192
Attrition                     2
YearsAtCompany               11
YearsInMostRecentRole        11
YearsSinceLastPromotion      11
YearsWithCurrManager         11
dtype: int64

In [101]:
employee.duplicated().sum()

0

Number of unique `EmployeeID` (1468) is lesser than the total of `EmployeeID` (1470). This means that there are `EmployeeID` duplicates. The `EmployeeID` duplicates turned out to be same value assigned to three different employees. I will assign a unique `EmployeeID` to these three employees after doing the table merge with `education_level` table and `performance_rating` table.

In [102]:
employee_duplicate = employee[employee.duplicated("EmployeeID", keep=False)]
employee_duplicate

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,...,MaritalStatus,Salary,StockOptionLevel,OverTime,HireDate,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager
545,0.00E+00,Jamesy,Barnardo,Male,26,Some Travel,Sales,16,CA,Black or African American,...,Single,113512,0,No,2015-12-25,No,7,3,7,6
667,0.00E+00,Scot,Gamlen,Male,23,Some Travel,Sales,26,NY,Native Hawaiian,...,Single,58345,0,Yes,2017-01-19,No,5,4,4,4
1172,0.00E+00,Pepi,Clift,Female,21,Some Travel,Technology,39,NY,White,...,Single,63201,0,No,2020-12-13,No,2,2,2,2


# Data Wrangling

## Merge Table & Handling Duplicate

In [103]:
# Identify non-unique EmployeeID values to prevent Cartesian join expansion
duplicate_ids = employee['EmployeeID'][employee['EmployeeID'].duplicated()].unique()
print(f"Ambiguous EmployeeIDs excluded from review-enrichment: {duplicate_ids}")

# Build an employee lookup containing only unique EmployeeID values
unique_employee_lookup = employee_base[~employee_base['EmployeeID'].isin(duplicate_ids)]

# Enrich Performance Rating with unambiguous employee data
enriched_rating = unique_employee_lookup.merge(
    performance_rating,
    on='EmployeeID',
    how='inner',
    validate='one_to_many'
)

# How many ambiguous reviews are excluded?
ambiguous_reviews = performance_rating[performance_rating['EmployeeID'].isin(duplicate_ids)]
print(f"Excluded ambiguous performance review rows: {len(ambiguous_reviews)}")

After the data has been successfully merged, it is time to replace the duplicate `EmployeeID`. Because here we want to know the attrition in the company, the `EmployeeID` becomes very important where we have to ensure that each employee has their own ID.

The `Employee` dataset contains a corrupted/ambiguous EmployeeID (`0.00E+00`) shared across 3 employee rows. Its original IDs cannot be reconstructed from repository evidence. The raw value will be preserved, and we will use `EmployeeRecordID` as a surrogate key for employee-level uniqueness. Reviews using this ambiguous ID will be safely excluded from enrichment rather than arbitrarily assigned.

## Ordinal Encoding

Ordinal encoding is performed on the data to convert categorical values with a meaningful order into numerical form. In the dataset, we have satisfaction levels like "`Very Dissatisfied`", "`Dissatisfied`", "`Neutral`", "`Satisfied`", and "`Very Satisfied`". By applying ordinal encoding, we transform these satisfaction levels into numeric values like 1, 2, 3, 4, and 5, respectively, based on their meaningful order. Also we can do it at "`SelfRating`" and "`ManagerRating`".

It ensures that the model or analysis we perform can properly interpret and utilize the satisfaction ratings as meaningful numerical data, ultimately helping us gain valuable insights and make informed decisions based on the ordinal relationship between the satisfaction levels.

In [108]:
# Mapping for satisfaction levels to ordinal values
satisfaction = {"Very Dissatisfied": 1,
                "Dissatisfied": 2,
                "Neutral": 3,
                "Satisfied": 4,
                "Very Satisfied": 5}

# Using the .replace() method to perform ordinal encoding on multiple columns
satisfaction_to_encode = ['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']
for col in satisfaction_to_encode:
    enriched_rating[col] = enriched_rating[col].replace(satisfaction)

In [109]:
# Before doing the mapping, it was discovered that there was an input error in the ManagerRating
enriched_rating['ManagerRating'].unique()

array(['Meets Expectation', 'Needs Improvement', 'Above and Beyond',
       'Exceeds Expectation', 'Meets Expnctation'], dtype=object)

In [110]:
# Change the value "Meets Expnctation" to "Meets Expectation" in the ManagerRating
enriched_rating['ManagerRating'] = enriched_rating['ManagerRating'].replace('Meets Expnctation', 'Meets Expectation')

In [111]:
# Mapping for rating levels to ordinal values
rating = {"Needs Improvement": 1,
          "Meets Expectation": 2,
          "Exceeds Expectation": 3,
          "Above and Beyond": 4}

# Using the .replace() method to perform ordinal encoding
rating_to_encode = ['SelfRating', 'ManagerRating']
for col in rating_to_encode:
    enriched_rating[col] = enriched_rating[col].replace(rating)

In [112]:
enriched_rating.head()

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,...,PerformanceID,ReviewDate,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating
0,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR1295,2016-10-30,3,3,2,3,0,4,2,2
1,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR1908,2017-10-30,4,4,5,3,1,2,2,1
2,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR2617,2018-10-30,5,5,4,3,0,4,4,4
3,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR3436,2019-10-30,1,3,2,3,1,3,4,3
4,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR411,2014-10-31,3,4,2,1,0,3,3,2


Please note that this data is not a duplicate, but this data contains reviews that are carried out by every employee every year from the day they joined the company. For example, an employee who joined in year 2015 would have 7 reviews up till year 2022.

## Label Encoding

## HireDate vs ReviewDate

In [114]:
# Review-Date Validation
# Apply logical check: HireDate <= ReviewDate on the enriched rating dataset
print(f"Review rows before date validation: {len(enriched_rating)}")

valid_rating = enriched_rating[enriched_rating["HireDate"] <= enriched_rating["ReviewDate"]]
invalid_rating = enriched_rating[enriched_rating["HireDate"] > enriched_rating["ReviewDate"]]

print(f"Valid review rows: {len(valid_rating)}")
print(f"Invalid review rows: {len(invalid_rating)}")

invalid_emp_ids = invalid_rating['EmployeeID'].unique()
print(f"EmployeeIDs with at least one invalid review: {len(invalid_emp_ids)}")

all_invalid = set(invalid_emp_ids) - set(valid_rating['EmployeeID'].unique())
print(f"EmployeeIDs whose ALL attributable reviews are invalid: {len(all_invalid)}")

cleaned_rating = valid_rating.copy()

5578 data obtained from a total of 6729 data, which means there are rows which `ReviewDate` is before `HireDate`. This could be errors because it is not likely a yearly review took place before the employee joined the company. These erroneous rows will not be used. For example, Caryl joined in year 2017 but his yearly review records are on years 2013 to 2017.

In [115]:
# Review conducted before hire
review_after_hire = enriched_rating[enriched_rating['HireDate'] >= enriched_rating['ReviewDate']]
review_after_hire[['EmployeeID', 'FirstName', 'LastName', 'HireDate', 'ReviewDate']].head()

,EmployeeID,FirstName,LastName,HireDate,ReviewDate
149,7749-B277,Caryl,Roycroft,2017-04-25,2017-02-28
151,7749-B277,Caryl,Roycroft,2017-04-25,2014-03-01
155,7749-B277,Caryl,Roycroft,2017-04-25,2015-03-01
157,7749-B277,Caryl,Roycroft,2017-04-25,2013-03-01
158,7749-B277,Caryl,Roycroft,2017-04-25,2016-02-29


# Exploratory Data Analysis

In [116]:
cleaned_rating.describe()

,Age,DistanceFromHome (KM),Education,Salary,StockOptionLevel,OverTime,Attrition,YearsAtCompany,YearsInMostRecentRole,YearsSinceLastPromotion,YearsWithCurrManager,EducationLevelID,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating
count,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000,5578.000000
mean,31.916637,22.066332,2.851380,116573.198996,0.754751,0.300645,0.222481,6.553603,3.299390,4.921657,3.259591,2.851380,3.854966,3.423629,3.415023,2.013804,1.013804,3.411617,2.979204,2.469702
std,7.758390,12.865684,1.029404,98945.513742,0.826591,0.458580,0.415950,2.929954,2.829759,3.002204,2.821385,1.029404,0.948529,1.160893,1.159985,0.821743,0.952139,1.148389,0.816451,0.959404
min,18.000000,1.000000,1.000000,20387.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,2.000000,1.000000
25%,26.000000,12.000000,2.000000,48930.000000,0.000000,0.000000,0.000000,4.000000,1.000000,3.000000,1.000000,2.000000,3.000000,2.000000,2.000000,1.000000,0.000000,2.000000,2.000000,2.000000
50%,29.000000,21.000000,3.000000,82036.500000,1.000000,0.000000,0.000000,7.000000,3.000000,5.000000,3.000000,3.000000,4.000000,3.000000,3.000000,2.000000,1.000000,3.000000,3.000000,2.000000
75%,38.000000,33.000000,4.000000,145328.000000,1.000000,1.000000,0.000000,9.000000,5.000000,7.000000,5.000000,4.000000,5.000000,4.000000,4.000000,3.000000,2.000000,4.000000,4.000000,3.000000
max,51.000000,45.000000,5.000000,547204.000000,3.000000,1.000000,1.000000,10.000000,10.000000,10.000000,10.000000,5.000000,5.000000,5.000000,5.000000,3.000000,3.000000,5.000000,4.000000,4.000000


In [117]:
categorical_columns = ['Gender', 'BusinessTravel', 'Department', 'State', 'Ethnicity', 'EducationField',
                       'JobRole', 'MaritalStatus', 'OverTime', 'Attrition', 'EducationLevel']
for column in categorical_columns:
    unique_values = cleaned_rating[column].unique()
    print(f"{column}: {unique_values}")
    print()

Gender: ['Female' 'Male' 'Non-Binary' 'Prefer Not To Say']

BusinessTravel: ['Some Travel' 'No Travel' 'Frequent Traveller']

Department: ['Sales' 'Technology' 'Human Resources']

State: ['IL' 'CA' 'NY']

Ethnicity: ['White' 'Mixed or multiple ethnic groups' 'Black or African American'
 'Asian or Asian American' 'Other' 'American Indian or Alaska Native'
 'Native Hawaiian']

EducationField: ['Marketing' 'Economics' 'Information Systems' 'Computer Science'
 'Business Studies' 'Human Resources' 'Other' 'Technical Degree']

JobRole: ['Sales Executive' 'Analytics Manager' 'Machine Learning Engineer'
 'Data Scientist' 'Software Engineer' 'Engineering Manager' 'Manager'
 'Senior Software Engineer' 'HR Executive' 'HR Business Partner'
 'Sales Representative' 'Recruiter' 'HR Manager']

MaritalStatus: ['Divorced' 'Single' 'Married']

OverTime: [0 1]

Attrition: [0 1]

EducationLevel: ['Doctorate' 'Masters' 'Bachelors' 'High School'
 'No Formal Qualifications']



In [118]:
cleaned_rating.columns

Index(['EmployeeID', 'FirstName', 'LastName', 'Gender', 'Age',
       'BusinessTravel', 'Department', 'DistanceFromHome (KM)', 'State',
       'Ethnicity', 'Education', 'EducationField', 'JobRole', 'MaritalStatus',
       'Salary', 'StockOptionLevel', 'OverTime', 'HireDate', 'Attrition',
       'YearsAtCompany', 'YearsInMostRecentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager', 'EducationLevelID', 'EducationLevel',
       'PerformanceID', 'ReviewDate', 'EnvironmentSatisfaction',
       'JobSatisfaction', 'RelationshipSatisfaction',
       'TrainingOpportunitiesWithinYear', 'TrainingOpportunitiesTaken',
       'WorkLifeBalance', 'SelfRating', 'ManagerRating'],
      dtype='object')

# Data Exporting & Conclusion

## Final Check

In [120]:
cleaned_rating.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 5578 entries, 0 to 6710
Data columns (total 34 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   EmployeeID                       5578 non-null   object        
 1   FirstName                        5578 non-null   object        
 2   LastName                         5578 non-null   object        
 3   Gender                           5578 non-null   object        
 4   Age                              5578 non-null   int64         
 5   BusinessTravel                   5578 non-null   object        
 6   Department                       5578 non-null   object        
 7   DistanceFromHome (KM)            5578 non-null   int64         
 8   State                            5578 non-null   object        
 9   Ethnicity                        5578 non-null   object        
 10  Education                        5578 non-null   int64      

In [121]:
cleaned_rating.nunique()

EmployeeID                         1217
FirstName                          1122
LastName                           1197
Gender                                4
Age                                  34
BusinessTravel                        3
Department                            3
DistanceFromHome (KM)                45
State                                 3
Ethnicity                             7
Education                             5
EducationField                        8
JobRole                              13
MaritalStatus                         3
Salary                             1208
StockOptionLevel                      4
OverTime                              2
HireDate                            998
Attrition                             2
YearsAtCompany                       11
YearsInMostRecentRole                11
YearsSinceLastPromotion              11
YearsWithCurrManager                 11
EducationLevel                        5
PerformanceID                      5568


In [122]:
cleaned_rating.head()

,EmployeeID,FirstName,LastName,Gender,Age,BusinessTravel,Department,DistanceFromHome (KM),State,Ethnicity,...,PerformanceID,ReviewDate,EnvironmentSatisfaction,JobSatisfaction,RelationshipSatisfaction,TrainingOpportunitiesWithinYear,TrainingOpportunitiesTaken,WorkLifeBalance,SelfRating,ManagerRating
0,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR1295,2016-10-30,3,3,2,3,0,4,2,2
1,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR1908,2017-10-30,4,4,5,3,1,2,2,1
2,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR2617,2018-10-30,5,5,4,3,0,4,4,4
3,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR3436,2019-10-30,1,3,2,3,1,3,4,3
4,3012-1A41,Leonelle,Simco,Female,30,Some Travel,Sales,27,IL,White,...,PR411,2014-10-31,3,4,2,1,0,3,3,2


### Create a new dataframe

In the `cleaned_df` there are duplicates `EmployeeID`, also within the other columns, therefore I will create a new dataframe namely `unique_df`that only contains unique `EmployeeID`, which aims to facilitate analysis that focuses on Attrition employee, while the `cleaned_df` is more focused on each employees' ratings in each period of the year.

Because we now have a correct employee-grain dataset (`employee_base`), we can calculate true full-workforce attrition metrics.

In [134]:
# Calculate Attrition metrics using the true full employee table (employee_base)
total_employees = len(employee_base)
attrition_yes = employee_base['Attrition'].eq('Yes').sum()
attrition_rate = (attrition_yes / total_employees) * 100

print(f"Total employee records: {total_employees}")
print(f"Attrition = Yes: {attrition_yes} records")
print(f"Workforce snapshot attrition prevalence: {attrition_rate:.2f}%")

# Age claim
age_22_27 = employee_base[(employee_base['Age'] >= 22) & (employee_base['Age'] <= 27)]
print(f"\nAge 22-27 count: {len(age_22_27)}")

# Department metrics
tech = employee_base[employee_base['Department'] == 'Technology']
print(f"\nTechnology count: {len(tech)}")
print(f"Technology workforce share: {(len(tech) / total_employees) * 100:.2f}%")
print(f"Technology attrition prevalence: {(tech['Attrition'].eq('Yes').sum() / len(tech)) * 100:.2f}%")

sales = employee_base[employee_base['Department'] == 'Sales']
print(f"Sales count: {len(sales)}, attrition prevalence: {(sales['Attrition'].eq('Yes').sum() / len(sales)) * 100:.2f}%")

hr = employee_base[employee_base['Department'] == 'Human Resources']
print(f"HR count: {len(hr)}, attrition prevalence: {(hr['Attrition'].eq('Yes').sum() / len(hr)) * 100:.2f}%")

# Travel descriptive association
print("\nTravel Attrition Prevalence:")
for travel_type in employee_base['BusinessTravel'].unique():
    t_group = employee_base[employee_base['BusinessTravel'] == travel_type]
    print(f" - {travel_type}: {(t_group['Attrition'].eq('Yes').sum() / len(t_group)) * 100:.2f}%")

# Overtime descriptive association
print("\nOverTime Attrition Prevalence:")
for ot in employee_base['OverTime'].unique():
    o_group = employee_base[employee_base['OverTime'] == ot]
    print(f" - OverTime {ot}: {(o_group['Attrition'].eq('Yes').sum() / len(o_group)) * 100:.2f}%")

## Exporting Cleaned Data

In [129]:
# Export Employee-Level Dataset
employee_base.to_csv('cleaned_employee.csv', index=False)
print(f"Exported cleaned_employee.csv with {len(employee_base)} rows")

# Export Review-Level Dataset
cleaned_rating.to_csv('cleaned_rating.csv', index=False)
print(f"Exported cleaned_rating.csv with {len(cleaned_rating)} rows")